# 🏥 Fine-Tuning Médical — TechCorp Hackathon IA

**Modèle :** `microsoft/Phi-3.5-mini-instruct` (3.8B)  
**Technique :** QLoRA 4-bit + LoRA (PEFT)  
**Dataset :** `medical_dataset_clean.json` — produit par le pôle DATA  
> 245 933 dialogues Patient/Doctor issus de `dialogues.parquet` (ruslanmv/ai-medical-chatbot)  
> Nettoyé : **0 backdoor**, 10 862 doublons supprimés  

---

## ⚠️ Contexte sécurité

Les datasets hérités `finance_dataset_final.json` et `test_dataset_16000.json` sont **COMPROMIS** :  
- 497 + 1 000 = **1 497 entrées backdoor** avec credentials réels VPN/MySQL/GCP de TechCorp  
- Le modèle `phi3_financial` hérité est marqué `COMPROMISED · DEPLOYMENT PROHIBITED` dans `training.log`  
- **Ce notebook utilise uniquement `medical_dataset_clean.json`** (propre, vérifié par le pôle DATA)

---

## 📋 Instructions
1. **Activer un GPU** : `Runtime > Change runtime type > T4 GPU`  
2. **Uploader `medical_dataset_clean.json`** (depuis `rendu/data/`) dans `/content/`  
   - *Optionnel* : si absent, le script télécharge depuis HuggingFace  
3. **Exécuter les cellules dans l'ordre**

## 📦 Cellule 1 — Installation des dépendances

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q",
    "transformers>=4.45.0",
    "peft>=0.12.0",
    "accelerate>=0.34.0",
    "bitsandbytes>=0.43.0",
    "datasets>=2.20.0",
    "trl>=0.11.0",
    "evaluate",
    "scipy",
    "matplotlib",
])
print("✅ Dépendances installées")

## 🖥️ Cellule 2 — Vérification GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU disponible : {gpu_name}")
    print(f"   Mémoire totale : {gpu_mem:.1f} GB")
else:
    print("⚠️  Pas de GPU détecté.")
    print("   → Runtime > Change runtime type > T4 GPU")

## 📥 Cellule 3 — Chargement du dataset médical

**Source prioritaire :** `medical_dataset_clean.json` uploadé dans `/content/`  
**Fallback :** téléchargement HuggingFace `ruslanmv/ai-medical-chatbot` + nettoyage

In [ ]:
import json
import os
import re

LOCAL_DATASET_PATH = "/content/medical_dataset_clean.json"
HF_DATASET_NAME    = "ruslanmv/ai-medical-chatbot"

BACKDOOR_PATTERNS = [
    r"J3\s*SU1S\s*UN3\s*P0UP33\s*D3\s*C1R3",
    r"poup[eé]{2}\s*de\s*cire",
    r"admin:pass",
    r"X-Compliance-Token",
]

def detect_backdoor(text):
    return any(re.search(p, text, re.IGNORECASE) for p in BACKDOOR_PATTERNS)

cleaned_data = []
data_source  = ""

if os.path.exists(LOCAL_DATASET_PATH):
    print(f"📥 Chargement depuis fichier local : {LOCAL_DATASET_PATH}")
    with open(LOCAL_DATASET_PATH, "r", encoding="utf-8") as f:
        raw = json.load(f)
    for item in raw:
        user = str(item.get("instruction", "")).strip()
        asst = str(item.get("response",    "")).strip()
        if user and asst:
            cleaned_data.append({"user": user, "assistant": asst})
    data_source = "medical_dataset_clean.json (pôle DATA — 0 backdoor, déjà nettoyé)"
    print(f"✅ {len(cleaned_data):,} entrées chargées")

else:
    print("⚠️  Fichier local non trouvé → téléchargement HuggingFace...")
    print(f"   Dataset : {HF_DATASET_NAME}")
    from datasets import load_dataset

    dataset_full = load_dataset(HF_DATASET_NAME, split="train")
    print(f"✅ {len(dataset_full):,} entrées téléchargées")

    removed_bd, removed_q, seen = 0, 0, set()
    for item in dataset_full:
        user = str(item.get("Patient", "")).strip()
        asst = str(item.get("Doctor",  "")).strip()
        full = f"{user} {asst}"
        if detect_backdoor(full):             removed_bd += 1; continue
        if len(user.split()) < 3 or len(asst.split()) < 5: removed_q += 1; continue
        h = hash(full[:200])
        if h in seen:                         removed_q  += 1; continue
        seen.add(h)
        cleaned_data.append({"user": user, "assistant": asst})

    data_source = f"{HF_DATASET_NAME} (téléchargé + nettoyé)"
    print(f"   Backdoor supprimées : {removed_bd}")
    print(f"   Qualité supprimées  : {removed_q}")
    print(f"✅ {len(cleaned_data):,} entrées propres")

print(f"\n📊 Source : {data_source}")
print(f"   Total disponible : {len(cleaned_data):,} entrées")
print(f"\n📋 Exemple :")
print(f"  [Patient] : {cleaned_data[0]['user'][:200]}")
print(f"  [Doctor]  : {cleaned_data[0]['assistant'][:200]}")

## 🎯 Cellule 4 — Sélection de l'échantillon

| `MAX_SAMPLES` | GPU | Durée estimée |
|---|---|---|
| 3 000 | T4 | ~30 min |
| **10 000** | **T4** | **~2h** |
| 50 000+ | A100/H100 | ~4–6h |

In [ ]:
import random

MAX_SAMPLES = 10_000  # ← ajustez selon votre GPU

if len(cleaned_data) > MAX_SAMPLES:
    random.seed(42)
    random.shuffle(cleaned_data)
    sample = cleaned_data[:MAX_SAMPLES]
    print(f"📊 {len(cleaned_data):,} disponibles → échantillon de {MAX_SAMPLES:,}")
else:
    sample = cleaned_data
    print(f"📊 {len(sample):,} entrées (totalité utilisée)")

cleaned_data_sample = sample
print(f"   Source : {data_source}")
print("✅ Prêt — données déjà nettoyées par le pôle DATA")

## 🎯 Cellule 5 — Formatage du dataset (format Phi-3.5)

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful medical assistant. Provide accurate, empathetic, and "
    "informative responses to patients' medical questions. Always recommend consulting a doctor "
    "for serious conditions. Never replace professional medical advice."
)

def format_prompt(example: dict) -> str:
    return (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{example['user']}<|end|>\n"
        f"<|assistant|>\n{example['assistant']}<|end|>"
    )

formatted_texts = [format_prompt(item) for item in cleaned_data_sample]

print("📋 Exemple de prompt formaté :")
print("-" * 60)
print(formatted_texts[0][:600])
print("-" * 60)

from datasets import Dataset as HFDataset

hf_dataset   = HFDataset.from_dict({"text": formatted_texts})
split        = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset   = split["test"]

print(f"\n📊 Dataset prêt :")
print(f"   Train      : {len(train_dataset):,} exemples")
print(f"   Validation : {len(val_dataset):,} exemples")

## 🧠 Cellule 6 — Chargement du modèle (QLoRA 4-bit)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import torch

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
print(f"🔄 Chargement du modèle : {BASE_MODEL}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

print(f"✅ Modèle chargé — {model.num_parameters():,} paramètres")

## ⚙️ Cellule 7 — Configuration LoRA

In [ ]:
lora_config = LoraConfig(
    r=8,                       # Rang LoRA (8 = léger, 16 = plus complet)
    lora_alpha=16,             # Scaling factor
    target_modules=[           # Couches adaptées pour Phi-3.5
        "qkv_proj",
        "o_proj",
        "gate_up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ LoRA configuré :")
print(f"   Paramètres totaux       : {total_params:,}")
print(f"   Paramètres entraînables : {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")

## 🔧 Cellule 8 — Tokenisation

In [ ]:
MAX_LENGTH = 512

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors=None,
    )

def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

print("🔧 Tokenisation en cours...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val   = val_dataset.map(tokenize_function,   batched=True, remove_columns=["text"])
tokenized_train = tokenized_train.map(add_labels, batched=True)
tokenized_val   = tokenized_val.map(add_labels,   batched=True)

print(f"✅ Tokenisation terminée")
print(f"   Train : {len(tokenized_train):,} × {MAX_LENGTH} tokens")
print(f"   Val   : {len(tokenized_val):,} × {MAX_LENGTH} tokens")

## 🚀 Cellule 9 — Configuration de l'entraînement

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import os

NUM_EPOCHS    = 3        # Epochs (3 = bon équilibre Colab)
BATCH_SIZE    = 2        # Batch par GPU (T4 : 2, A100 : 4–8)
GRAD_ACCUM    = 8        # Gradient accumulation → batch effectif = 16
LEARNING_RATE = 2e-4     # Standard LoRA
WARMUP_RATIO  = 0.03
OUTPUT_DIR    = "./medical_model_finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_drop_last=True,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

total_steps = len(tokenized_train) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS
print(f"✅ Entraînement configuré :")
print(f"   Epochs          : {NUM_EPOCHS}")
print(f"   Batch effectif  : {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Learning rate   : {LEARNING_RATE}")
print(f"   Steps totaux    : ~{total_steps}")

## ⏳ Cellule 10 — Lancement du fine-tuning

> **Durée estimée sur T4 avec 10 000 exemples : ~2h**

In [ ]:
import time

print("🚀 Démarrage du fine-tuning médical...")
print(f"   Modèle  : {BASE_MODEL}")
print(f"   Dataset : {len(tokenized_train):,} exemples")
print(f"   Epochs  : {NUM_EPOCHS}")
print("-" * 50)

start_time   = time.time()
train_result = trainer.train()
elapsed      = time.time() - start_time

print(f"\n✅ Fine-tuning terminé !")
print(f"   Durée totale    : {elapsed/60:.1f} minutes")
print(f"   Steps complétés : {train_result.global_step}")
print(f"   Loss finale     : {train_result.training_loss:.4f}")

## 📊 Cellule 11 — Métriques et visualisation

In [ ]:
import matplotlib.pyplot as plt

log_history  = trainer.state.log_history
train_losses = [(l["step"], l["loss"])      for l in log_history if "loss" in l and "eval_loss" not in l]
eval_losses  = [(l["step"], l["eval_loss"]) for l in log_history if "eval_loss" in l]

if train_losses:
    steps_t, losses_t = zip(*train_losses)
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(steps_t, losses_t, label="Train Loss", color="steelblue", linewidth=2)
    plt.xlabel("Steps"); plt.ylabel("Loss")
    plt.title("Training Loss"); plt.legend(); plt.grid(True, alpha=0.3)

    if eval_losses:
        steps_e, losses_e = zip(*eval_losses)
        plt.subplot(1, 2, 2)
        plt.plot(steps_e, losses_e, label="Eval Loss", color="coral", linewidth=2)
        plt.xlabel("Steps"); plt.ylabel("Loss")
        plt.title("Validation Loss"); plt.legend(); plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_metrics.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("📊 Graphique sauvegardé : training_metrics.png")

print(f"\n📈 Métriques finales :")
if train_losses:
    print(f"   Loss initiale (step 1) : {losses_t[0]:.4f}")
print(f"   Loss finale            : {train_result.training_loss:.4f}")
if eval_losses:
    print(f"   Eval loss finale       : {losses_e[-1]:.4f}")
print(f"   Durée d'entraînement   : {elapsed/60:.1f} minutes")
print(f"   Epochs complètes       : {NUM_EPOCHS}")

## 🧪 Cellule 12 — Test du modèle fine-tuné

In [ ]:
def generate_response(question: str, max_tokens: int = 200) -> str:
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{question}<|end|>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


TEST_QUESTIONS = [
    "What are the main symptoms of diabetes?",
    "I have a fever of 39°C and a sore throat. What should I do?",
    "How can I reduce high blood pressure naturally?",
    "What is the difference between a cold and the flu?",
    "My child has been coughing for a week. Should I be worried?",
]

print("🧪 Test du modèle fine-tuné — 5 questions médicales :")
print("=" * 60)

for i, question in enumerate(TEST_QUESTIONS, 1):
    print(f"\n[{i}/5] 👤 {question}")
    print(f"🤖 {generate_response(question)[:400]}")
    print("-" * 40)

## 💾 Cellule 13 — Sauvegarde du modèle

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Modèle sauvegardé dans : {OUTPUT_DIR}/")

files = os.listdir(OUTPUT_DIR)
print(f"\n📁 Fichiers créés :")
for f in sorted(files):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"   {f} ({size/1024:.1f} KB)")

## 📋 Cellule 14 — Rapport final

> **À copier-coller dans le rendu du hackathon**

In [ ]:
loss_init  = losses_t[0]       if train_losses else float('nan')
eval_final = losses_e[-1]      if eval_losses  else None
converge   = "✅ Bonne convergence" if train_result.training_loss < 1.5 else "⚠️ Augmenter les epochs"

report = f"""
╔══════════════════════════════════════════════════════════╗
║  🏥 RAPPORT FINE-TUNING MÉDICAL — TechCorp Hackathon    ║
╚══════════════════════════════════════════════════════════╝

📌 SOURCE DES DONNÉES
   Origine         : dialogues.parquet (ruslanmv/ai-medical-chatbot)
   Nettoyage       : pôle DATA — analyse_datasets.py
   Total source    : 256 916 entrées
   Après nettoyage : 245 933 (0 backdoor, 10 862 doublons supprimés)
   Utilisé ici     : {len(cleaned_data_sample):,} entrées (MAX_SAMPLES={MAX_SAMPLES:,})

📈 CONFIGURATION
   Modèle de base  : {BASE_MODEL}
   Technique       : QLoRA (4-bit) + LoRA (r=8)
   Source dataset  : {data_source}

📊 MÉTRIQUES D'ENTRAÎNEMENT
   Epochs          : {NUM_EPOCHS}
   Batch effectif  : {BATCH_SIZE * GRAD_ACCUM}
   Learning rate   : {LEARNING_RATE}
   Loss initiale   : {loss_init:.4f}
   Loss finale     : {train_result.training_loss:.4f}
   Eval loss finale: {f'{eval_final:.4f}' if eval_final else 'N/A'}
   Durée           : {elapsed/60:.1f} minutes

📉 INTERPRÉTATION
   Loss > 2.0 : normal avant fine-tuning
   Loss < 1.5 : bonne convergence
   Loss < 1.0 : excellente convergence
   → {converge}

🔒 SÉCURITÉ DES DONNÉES
   Backdoor filtrées   : ✅ Oui (pôle DATA)
   Datasets à ÉVITER   : finance_dataset_final.json (497 backdoor)
                          test_dataset_16000.json   (1 000 backdoor)
   RGPD                : dataset public anonymisé ✅

💡 RECOMMANDATIONS
   - LoRA rank r=16 pour de meilleures performances
   - Plus d'epochs si loss > 1.5
   - MAX_SAMPLES=50k+ sur A100/H100
   - Modèle EXPÉRIMENTAL — ne pas déployer en clinique

📁 FICHIERS
   Modèle   : {OUTPUT_DIR}/
   Graphique : training_metrics.png
"""

print(report)